# Tutorial: control one shutter with Python

Learn to select a KSC101, connect, request Close/Open, read feedback, measure API
call durations, and release the connection. Start with one cycle; an optional
exercise retains the earlier 20-cycle experiment.

## 1. Prepare the session

Select the project's `.venv` Python kernel. Restart it after editing the controller
or project configuration. Close the GUI, Kinesis, and other device owners.
Match the selected serial to the physical shutter. Establish independent source
protection, the normal physical interlock, and approved operating conditions.
The operation cells below move real hardware; imports and settings are passive.
See [commissioning guidance](../HARDWARE_VALIDATION.md) for bench prerequisites.

## 2. Select the controller and hold time

The serial and SDK folder default to project `config.py`. Override `SERIAL` below
if needed: both strings (`"68000001"`) and integers (`68000001`) are accepted.
The example identity is fictional. No command-line arguments are needed.

`HOLD_SECONDS` is an added wait after Open feedback, in seconds. Choose an approved
duration. Users manage the physical interlock externally; no software interlock
confirmation is required.

In [ ]:
from contextlib import contextmanager
from math import isfinite
from time import perf_counter, sleep

from thorlabs_shutter_control import KCubeShutter, config

SERIAL = config.SERIAL_NUMBERS[0] if config.SERIAL_NUMBERS else ""  # Or enter the serial.
HOLD_SECONDS = 1.0  # Choose an approved added hold, in seconds.

## 3. Validate settings

Check the single-cycle hold before connecting. Optional repetition settings and
validation are kept together in section 8.

In [ ]:
if not isfinite(HOLD_SECONDS) or HOLD_SECONDS <= 0:
    raise ValueError("HOLD_SECONDS must be a finite positive duration.")

## 4. Prepare the controller and error cleanup

Constructing the controller is passive. `session_step()` attempts Close and release
if a hardware cell raises an exception or is interrupted. Close is permitted only
after Connect verifies identity. Successful cells keep the connection for the next
step. Run cells in order, and always finish with section 9 to disconnect.

**Between the Open and Close cells the shutter may remain open, including while
you pause to read.** The hold is not a maximum exposure bound. Use independent
source protection; run Close promptly. Skipping cleanup or terminating the kernel
can leave hardware active. Rerunning setup while connected can lose the handle;
disconnect before restarting this tutorial.

In [ ]:
shutter = KCubeShutter(SERIAL)
connected = False


@contextmanager
def session_step():
    try:
        yield
    except BaseException:
        shutter.disconnect(close_shutter=connected)
        raise

## 5. Connect and establish Closed

Connect verifies the serial without enabling the channel. The next cell explicitly
enables once and sends Disarm (Inactive, then Manual). Later Open/Close preserve
mode and enable state. Position feedback is requested separately.

In [ ]:
with session_step():
    print("Connected:", shutter.connect().serial_number)
    connected = True

In [ ]:
with session_step():
    shutter.enable_device()  # Explicit one-time SDK channel setup.
    shutter.disarm_triggers()
    print("Initial Close:", shutter.get_shutter_state())

## 6. Check readiness, then Open and hold

The caller checks Closed/Inactive/Manual and key before Open.
Open sends Active only; it preserves mode and enable state. Its timing excludes feedback and the added hold.
On success this cell leaves the connection active and the shutter commanded Open.
Run the Close cell next.

In [ ]:
with session_step():
    ready = shutter.get_operating_status()
    if not ready.is_closed_manual or not ready.key_enabled:
        raise RuntimeError("Verify Closed/Manual and key before Open")

    start = perf_counter()
    shutter.open_shutter()
    open_seconds = perf_counter() - start
    print(f"Open: {shutter.get_shutter_state()} ({open_seconds:.3f} s API call)")
    sleep(HOLD_SECONDS)

## 7. Close and read feedback

Close sends Inactive only; it preserves mode. Setters return `None`; the explicit getter requests
position feedback afterward. Independently observe physical closure.

In [ ]:
with session_step():
    start = perf_counter()
    shutter.close_shutter()
    close_seconds = perf_counter() - start
    print(f"Close: {shutter.get_shutter_state()} ({close_seconds:.3f} s API call)")

## 8. Optional: repeat Open/Close

This separate exercise runs **additional** cycles after the single cycle above.
It does nothing unless `RUN_REPEAT` is true. The earlier 20-cycle experiment with
0.04 s waits is retained below; these are not qualified physical limits.
Confirm the count and dwell suit your approved conditions before enabling it.

Each cycle checks readiness, opens, waits, closes, and waits again. Reads add
latency, so this is not a calibrated 40 ms pulse generator. The loop attempts Close
on normal completion or interruption; errors also release the connection.

Edit and run the settings cell here before the repetition cell. Leave `RUN_REPEAT`
false to skip this exercise. Validation occurs inside the error-cleanup guard.

In [ ]:
# Optional repetition exercise: enable only for this exercise.
RUN_REPEAT = False
REPEAT_COUNT = 20
REPEAT_HOLD_SECONDS = 0.04  # Earlier experiment's wait; not a qualified physical limit.

In [ ]:
with session_step():
    if not isinstance(RUN_REPEAT, bool):
        raise ValueError("RUN_REPEAT must be True or False.")
    if RUN_REPEAT:
        if isinstance(REPEAT_COUNT, bool) or not isinstance(REPEAT_COUNT, int) or REPEAT_COUNT <= 0:
            raise ValueError("REPEAT_COUNT must be a positive integer.")
        if not isfinite(REPEAT_HOLD_SECONDS) or REPEAT_HOLD_SECONDS <= 0:
            raise ValueError("REPEAT_HOLD_SECONDS must be a finite positive duration.")
    if RUN_REPEAT:
        try:
            ready = shutter.get_operating_status()
            if not ready.is_closed_manual or not ready.key_enabled:
                raise RuntimeError("Verify Closed/Manual and key before Open")
            for cycle_index in range(REPEAT_COUNT):
                print(f"Cycle {cycle_index + 1}/{REPEAT_COUNT}")
                shutter.open_shutter()
                sleep(REPEAT_HOLD_SECONDS)
                shutter.close_shutter()
                sleep(REPEAT_HOLD_SECONDS)
        finally:
            shutter.close_shutter()

## 9. Disconnect

Run this cell to finish, or to stop after any connected step. It attempts Close
only for a verified identity, then releases communication even if Close fails.
Release does not prove physical closure. Verify the shutter independently.

In [ ]:
shutter.disconnect(close_shutter=connected)
connected = False
print("Connection released. Independently verify physical closure.")

## 10. Interpret the results

Expected controller feedback is `Initial Close: closed`, `Open: open`, then
`Close: closed`, followed by connection release. A mismatch or exception is a
reason to stop and diagnose; a completed write alone does not establish motion.

Printed durations measure each setter call only. Feedback reads, holds, USB
latency outside that call, and mechanical travel are separate. A displayed
`0.000 s` is rounded, not instantaneous operation. Controller feedback and API
timing do not establish optical state or calibrated shutter timing.

The added hold is not a maximum exposure bound. Observe closure independently.
Cleanup attempts Close and release but cannot guarantee physical closure. Forced
kernel termination can skip cleanup. After an error, preserve the traceback,
verify physical state, resolve ownership or readiness, then rerun from settings.